In [ ]:
import pandas as pd

df = pd.read_csv("it_tickets.csv")

df.head()

,TicketID,TicketText,Category,Resolution,HasScreenshot,OCRText
0,1,Database server is unavailable,Database,"Check database connectivity, server status, cr...",0,NaN
1,2,Cannot send email Urgent issue.,Email,"Check mailbox storage, email settings, interne...",1,Access denied
2,3,Printer is not printing,Printer,"Check printer connection, paper/ink status, cl...",0,NaN
3,4,I received a suspicious email,Security,"Do not open suspicious links, run a security s...",0,NaN
4,5,Possible malware detected Need quick support.,Security,"Do not open suspicious links, run a security s...",0,NaN


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   TicketID       12000 non-null  int64 
 1   TicketText     12000 non-null  object
 2   Category       12000 non-null  object
 3   Resolution     12000 non-null  object
 4   HasScreenshot  12000 non-null  int64 
 5   OCRText        1202 non-null   object
dtypes: int64(2), object(4)
memory usage: 562.6+ KB


In [ ]:
print(df.shape)

(12000, 6)


In [ ]:
print(df.isnull().sum())

TicketID             0
TicketText           0
Category             0
Resolution           0
HasScreenshot        0
OCRText          10798
dtype: int64


In [ ]:
df["OCRText"] = df["OCRText"].fillna("")
print(df.isnull().sum())

TicketID         0
TicketText       0
Category         0
Resolution       0
HasScreenshot    0
OCRText          0
dtype: int64


In [ ]:
print(df["Category"].value_counts())

Category
Software          1583
Account Access    1552
Network           1508
Email             1503
Database          1484
Hardware          1465
Security          1457
Printer           1448
Name: count, dtype: int64


In [ ]:
print(df[["TicketText", "Category"]].head(10))


                                      TicketText        Category
0                 Database server is unavailable        Database
1                Cannot send email Urgent issue.           Email
2                        Printer is not printing         Printer
3                  I received a suspicious email        Security
4  Possible malware detected Need quick support.        Security
5               Software crashes when I start it        Software
6    Computer is overheating This started today.        Hardware
7     Query is running very slowly Urgent issue.        Database
8                    Unable to access my account  Account Access
9                     Database connection failed        Database


In [ ]:
X = df["TicketText"]
y = df["Category"]

print(X.shape)
print(y.shape)

(12000,)
(12000,)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(9600,)
(2400,)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

(9600, 81)
(2400, 81)


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
y_pred = model.predict(X_test_tfidf)

print(y_pred[:10])

['Email' 'Security' 'Network' 'Email' 'Hardware' 'Software'
 'Account Access' 'Email' 'Printer' 'Network']


In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 1.0


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

                precision    recall  f1-score   support

Account Access       1.00      1.00      1.00       310
      Database       1.00      1.00      1.00       297
         Email       1.00      1.00      1.00       300
      Hardware       1.00      1.00      1.00       293
       Network       1.00      1.00      1.00       302
       Printer       1.00      1.00      1.00       290
      Security       1.00      1.00      1.00       291
      Software       1.00      1.00      1.00       317

      accuracy                           1.00      2400
     macro avg       1.00      1.00      1.00      2400
  weighted avg       1.00      1.00      1.00      2400



In [ ]:
# model achieved 100% accuracy on the synthetic test dataset; real-world performance must be validated on actual historical IT tickets.”

In [ ]:
import time

start = time.time()

prediction = model.predict(X_test_tfidf)

end = time.time()

print("Prediction Time:", end - start, "seconds")

Prediction Time: 0.001245260238647461 seconds


In [ ]:
df["CombinedText"] = df["TicketText"] + " " + df["OCRText"]

print(df[["TicketText", "OCRText", "CombinedText"]].head())

                                      TicketText        OCRText  \
0                 Database server is unavailable                  
1                Cannot send email Urgent issue.  Access denied   
2                        Printer is not printing                  
3                  I received a suspicious email                  
4  Possible malware detected Need quick support.                  

                                     CombinedText  
0                 Database server is unavailable   
1   Cannot send email Urgent issue. Access denied  
2                        Printer is not printing   
3                  I received a suspicious email   
4  Possible malware detected Need quick support.   


In [ ]:
X = df["CombinedText"]
y = df["Category"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)


(9600,)
(2400,)


In [ ]:
vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

(9600, 87)
(2400, 87)


In [ ]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
y_pred = model.predict(X_test_tfidf)

print(y_pred[:10])

['Email' 'Security' 'Network' 'Email' 'Hardware' 'Software'
 'Account Access' 'Email' 'Printer' 'Network']


In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 1.0


In [ ]:
solution_map = df.groupby("Category")["Resolution"].first().to_dict()

print(solution_map)

{'Account Access': 'Verify username, reset the password, and unlock the account if necessary.', 'Database': 'Check database connectivity, server status, credentials, and query performance.', 'Email': 'Check mailbox storage, email settings, internet connection, and restart the email client.', 'Hardware': 'Check hardware connections, restart the device, and escalate for hardware inspection if needed.', 'Network': 'Check network connection, restart network adapter, and verify router/switch connectivity.', 'Printer': 'Check printer connection, paper/ink status, clear stuck jobs, and restart the printer.', 'Security': 'Do not open suspicious links, run a security scan, and escalate the incident to the security team.', 'Software': 'Restart the application, check installation, and reinstall or update the software if required.'}


In [ ]:
def auto_resolve(ticket):
    ticket_tfidf = vectorizer.transform([ticket])
    predicted_category = model.predict(ticket_tfidf)[0]
    solution = solution_map[predicted_category]

    return predicted_category, solution

In [ ]:
category, solution = auto_resolve("My laptop cannot connect to the office WiFi")

print("Predicted Category:", category)
print("Suggested Solution:", solution)

Predicted Category: Network
Suggested Solution: Check network connection, restart network adapter, and verify router/switch connectivity.


In [ ]:
import time

start = time.time()

category, solution = auto_resolve(
    "My laptop cannot connect to the office WiFi"
)

end = time.time()

print("Category:", category)
print("Solution:", solution)
print("Response Time:", end - start, "seconds")

Category: Network
Solution: Check network connection, restart network adapter, and verify router/switch connectivity.
Response Time: 0.0034737586975097656 seconds


In [ ]:
batch_size = 10000

print("Tickets:", 2_000_000)
print("Batch size:", batch_size)
print("Number of batches:", 2_000_000 // batch_size)

Tickets: 2000000
Batch size: 10000
Number of batches: 200


In [ ]:
batch_size = 1000

for start in range(0, len(df), batch_size):
    batch = df.iloc[start:start + batch_size]
    print("Processing tickets:", start, "to", start + len(batch) - 1)
    break

Processing tickets: 0 to 999


In [ ]:
def auto_resolve_with_confidence(ticket):
    ticket_tfidf = vectorizer.transform([ticket])

    predicted_category = model.predict(ticket_tfidf)[0]
    probabilities = model.predict_proba(ticket_tfidf)[0]

    confidence = max(probabilities)

    solution = solution_map[predicted_category]

    return predicted_category, confidence, solution

In [ ]:
category, confidence, solution = auto_resolve_with_confidence(
    "My laptop cannot connect to the office WiFi"
)

print("Category:", category)
print("Confidence:", confidence)
print("Solution:", solution)

Category: Network
Confidence: 0.8381333684854233
Solution: Check network connection, restart network adapter, and verify router/switch connectivity.


In [ ]:
def final_auto_resolve(ticket):
    ticket_tfidf = vectorizer.transform([ticket])

    predicted_category = model.predict(ticket_tfidf)[0]
    probabilities = model.predict_proba(ticket_tfidf)[0]
    confidence = max(probabilities)

    if confidence < 0.70:
        return predicted_category, confidence, "Escalate to IT support team."

    solution = solution_map[predicted_category]

    return predicted_category, confidence, solution

In [ ]:
category, confidence, solution = final_auto_resolve(
    "My laptop cannot connect to the office WiFi"
)

print("Category:", category)
print("Confidence:", confidence)
print("Action:", solution)

Category: Network
Confidence: 0.8381333684854233
Action: Check network connection, restart network adapter, and verify router/switch connectivity.
